<a href="https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deetijasmitha/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()

print("HF_TOKEN loaded successfully.")

HF_TOKEN loaded successfully.


In [26]:
from huggingface_hub import hf_hub_download

test_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("SUCCESS")
print(test_file)

SUCCESS
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [27]:
import duckdb

con = duckdb.connect()

print("DuckDB connection created.")

DuckDB connection created.


In [28]:
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face token configured for DuckDB.")

Hugging Face token configured for DuckDB.


In [29]:
test_query = """
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 1
"""

test_df = con.sql(test_query).df()

print("Warehouse connection successful.")
display(test_df)

Warehouse connection successful.


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

For Lane 2 — Refresh / Content Opportunity Scoring, one row represents the performance of one content item for one client on one report date.

I use the `fact_content_daily_performance` table as the main source.

I use March 2026 as the development window because it is a mid-panel month rather than the final `_sample` month.

The analysis supports prioritizing content pages for human review for possible refresh, improvement, protection, pruning, or monitoring.

In [30]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Table: fact_content_daily_performance")
print("Development month: 2026-03")
print("Expected grain: one row per report date, client, and content item")

Lane: Refresh / Content Opportunity Scoring
Table: fact_content_daily_performance
Development month: 2026-03
Expected grain: one row per report date, client, and content item


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

### Features

1. `gsc_impressions` — observed search exposure available at the decision moment.
2. `gsc_clicks` — observed search clicks available at the decision moment.
3. `gsc_avg_position` — observed search ranking position available at the decision moment.
4. `ga4_engaged_sessions` — observed engaged sessions available at the decision moment.
5. `ga4_total_engagement_sec` — observed engagement time available at the decision moment.

### Label / proxy

The future label is `is_declining_label`. It is calculated using April 2026 as the future outcome window: a content item receives label 1 when its total April GSC clicks are lower than its total March GSC clicks; otherwise it receives label 0. April is used only for the future outcome, while the selected features come from the March decision period. This keeps the feature and label windows separate and reduces the risk of target leakage.

### Context

- `report_date` — identifies when the observation was recorded.
- `month` — identifies the warehouse month.
- `client_hash_id` — identifies the anonymized client.
- `content_hash_id` — identifies the anonymized content item.
- `gsc_data_available` — indicates whether GSC data is available.
- `ga4_data_available` — indicates whether GA4 data is available.

### Excluded

I exclude fields that directly contain or are derived from the future outcome being predicted. I also exclude private or identifying information such as URLs, client names, and private search queries. A field is only used as a feature if it would have been available at the decision moment.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [31]:
query1 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        report_date || '|' || client_hash_id || '|' || content_hash_id
    ) AS unique_page_day_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print(con.sql(query1))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬──────────────────────┐
│ total_rows │ unique_page_day_rows │
│   int64    │        int64         │
├────────────┼──────────────────────┤
│    9841378 │              9841378 │
└────────────┴──────────────────────┘



The grain check confirms that all 9,841,378 rows have a unique combination of `report_date`, `client_hash_id`, and `content_hash_id`, with 0 duplicate grain rows.

In [32]:
query2 = """
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print(con.sql(query2))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [33]:
query3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print(con.sql(query3))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │
│   int64    │       int64        │
├────────────┼────────────────────┤
│    9841378 │            3611061 │
└────────────┴────────────────────┘



In [34]:
query3_result = con.sql(query3).df()

availability_rate = (
    query3_result["gsc_available_rows"].iloc[0]
    / query3_result["total_rows"].iloc[0]
)

print("GSC availability rate:", round(availability_rate, 3))

GSC availability rate: 0.367


### Five features

The five selected features are observable at the decision moment and describe search exposure, search response, ranking position, and engagement.

1. `gsc_impressions` — available because search impressions observed up to the decision date are known.
2. `gsc_clicks` — available because search clicks observed up to the decision date are known.
3. `gsc_avg_position` — available because average search position observed up to the decision date is known.
4. `ga4_engaged_sessions` — available because engaged sessions observed up to the decision date are known.
5. `ga4_total_engagement_sec` — available because engagement time observed up to the decision date is known.

In [35]:
features_query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_engaged_sessions,
    ga4_total_engagement_sec
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
LIMIT 10
"""

feature_df = con.sql(features_query).df()

print("Rows shown:", len(feature_df))
print("Feature count:", 5)

display(feature_df)

Rows shown: 10
Feature count: 5


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,ga4_total_engagement_sec
0,2026-03-01,client_65de48885f4ef01b,content_5c80451459c29b4a,5,0,5.400000,0,0
1,2026-03-01,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,39,0,5.666667,0,0
2,2026-03-01,client_65de48885f4ef01b,content_e25ea7297a1dffd3,179,0,5.156425,0,0
3,2026-03-01,client_65de48885f4ef01b,content_6b0149a80607dac3,72,0,7.694444,0,0
4,2026-03-01,client_65de48885f4ef01b,content_62673eea26c31c17,3282,1,6.167885,0,0
5,2026-03-01,client_65de48885f4ef01b,content_872342e050545a12,39,0,6.538462,0,0
6,2026-03-01,client_65de48885f4ef01b,content_3c286ded8bd68120,88,1,8.431818,0,0
7,2026-03-01,client_65de48885f4ef01b,content_b2108e8fe3360fa6,40,1,5.300000,0,0
8,2026-03-01,client_65de48885f4ef01b,content_4c185d1c173cd53d,23,0,30.304348,0,0
9,2026-03-01,client_65de48885f4ef01b,content_bd07be40ea0d5f54,23,0,5.478261,0,0


In [36]:
schema_query = """
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

schema_df = con.sql(schema_query).df()

display(schema_df)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 4. Data limits

The March 2026 development window contains 9,841,378 rows covering 55 clients and 331,437 content items from March 1 to March 31, 2026. Data availability is uneven: 3,611,061 rows have GSC data, while 413,966 have GA4 data, and only 364,347 rows have both sources available. There are 1,718,348 GSC-only rows and 49,619 GA4-only rows, so models using both sources will cover fewer observations. The March window is only one month, so it cannot represent the full historical behavior of every content item. Any future outcome label must use a later time period than the feature window to avoid time overlap and target leakage. The resulting score should therefore be treated as decision support for prioritizing content review, not as a definitive recommendation.

In [37]:
# Verify the expected grain:
# one row per report_date + client_hash_id + content_hash_id

grain_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR)
        || '|' ||
        client_hash_id
        || '|' ||
        content_hash_id
    ) AS unique_grain_rows,
    COUNT(*) - COUNT(DISTINCT
        CAST(report_date AS VARCHAR)
        || '|' ||
        client_hash_id
        || '|' ||
        content_hash_id
    ) AS duplicate_grain_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

grain_df = con.sql(grain_query).df()

display(grain_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_grain_rows,duplicate_grain_rows
0,9841378,9841378,0


The grain check confirms that all 9,841,378 rows have a unique combination of `report_date`, `client_hash_id`, and `content_hash_id`, with 0 duplicate grain rows.

In [38]:
# Section 4 — Data limits checks

limits_query = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
    ) AS both_available_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
        AND ga4_data_available IS FALSE
    ) AS gsc_only_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS FALSE
        AND ga4_data_available IS TRUE
    ) AS ga4_only_rows,

    COUNT(DISTINCT client_hash_id) AS unique_clients,

    COUNT(DISTINCT content_hash_id) AS unique_content_items,

    MIN(report_date) AS first_date,

    MAX(report_date) AS last_date

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

limits_df = con.sql(limits_query).df()

display(limits_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows,gsc_only_rows,ga4_only_rows,unique_clients,unique_content_items,first_date,last_date
0,9841378,3611061,413966,364347,1718348,49619,55,331437,2026-03-01,2026-03-31


In [39]:
# Check missing values for the five selected features

missing_features_query = """
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_impressions IS NULL
    ) AS missing_gsc_impressions,

    COUNT(*) FILTER (
        WHERE gsc_clicks IS NULL
    ) AS missing_gsc_clicks,

    COUNT(*) FILTER (
        WHERE gsc_avg_position IS NULL
    ) AS missing_gsc_avg_position,

    COUNT(*) FILTER (
        WHERE ga4_engaged_sessions IS NULL
    ) AS missing_ga4_engaged_sessions,

    COUNT(*) FILTER (
        WHERE ga4_total_engagement_sec IS NULL
    ) AS missing_ga4_total_engagement_sec

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""

missing_features_df = con.sql(missing_features_query).df()

display(missing_features_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_ga4_engaged_sessions,missing_ga4_total_engagement_sec
0,364347,0,0,0,0,0


In [40]:
# Check available warehouse months

months_query = """
SELECT
    month,
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=*/data_*.parquet'
)
GROUP BY month
ORDER BY month
"""

months_df = con.sql(months_query).df()

display(months_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,row_count,first_date,last_date
0,2025-01,1297,2025-01-27,2025-01-31
1,2025-02,75985,2025-02-01,2025-02-28
2,2025-03,167859,2025-03-01,2025-03-31
3,2025-04,285114,2025-04-01,2025-04-30
4,2025-05,349923,2025-05-01,2025-05-31
5,2025-06,329201,2025-06-01,2025-06-30
6,2025-07,469794,2025-07-01,2025-07-31
7,2025-08,704962,2025-08-01,2025-08-31
8,2025-09,845813,2025-09-01,2025-09-30
9,2025-10,2165471,2025-10-01,2025-10-31


The missing-value check shows 0 missing values for all five selected features among the 364,347 rows where both GSC and GA4 data are available. This means the selected modeling subset has complete values for these five features.

### Future label / proxy

The future label uses April 2026 as the outcome window. A content item is labeled as declining when its total GSC clicks in April are lower than its total GSC clicks in March. March is used for observed performance and April is used only for the future outcome, keeping the feature and label windows separate.

In [41]:
# Create the future decline proxy label
# March 2026 = observed period
# April 2026 = future outcome period

label_query = """
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS march_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.client_hash_id,
    m.content_hash_id,
    m.march_clicks,
    a.april_clicks,
    CASE
        WHEN a.april_clicks < m.march_clicks THEN 1
        ELSE 0
    END AS is_declining_label
FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
    AND m.content_hash_id = a.content_hash_id
"""

label_df = con.sql(label_query).df()

print("Label rows:", len(label_df))
display(label_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Label rows: 158549


,client_hash_id,content_hash_id,march_clicks,april_clicks,is_declining_label
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,1.0,0.0,1
1,client_62f4a7e64f5e0096,content_13a8105125458098,1.0,0.0,1
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,0.0,0.0,0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,1.0,1.0,0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,0.0,0.0,0


In [42]:
# Check future label distribution

label_distribution = (
    label_df["is_declining_label"]
    .value_counts()
    .sort_index()
)

print("Label distribution:")
print(label_distribution)

print("\nLabel proportions:")
print(
    label_df["is_declining_label"]
    .value_counts(normalize=True)
    .sort_index()
)

Label distribution:
is_declining_label
0    114244
1     44305
Name: count, dtype: int64

Label proportions:
is_declining_label
0    0.72056
1    0.27944
Name: proportion, dtype: float64


The April 2026 future-label check produces 158,549 labeled content items. Of these, 114,244 (72.06%) are labeled 0, meaning April clicks did not decline relative to March, and 44,305 (27.94%) are labeled 1, meaning clicks declined.

The label is imbalanced, with 27.94% declining and 72.06% not declining. Future modeling should account for this class distribution when evaluating performance.

## Self-check

- Every section above is filled — markdown thinking and the code that backs it.
- The notebook runs top to bottom with no errors.
- No client names, URLs, or private queries are included.
- Claims use careful words such as observed, measured, directional, and decision-support.
- The notebook is committed to the repository under `work/notebooks/`.